# 00 · Data preparation

Streams **FineWeb-Edu (sample-10BT)**, tokenises it with the Llama-2 SentencePiece tokenizer (32k vocab) and writes `train.bin` (51M tokens) + `val.bin` (1M tokens) as uint16 to Google Drive. Run once; notebooks 01–03 reuse the files.

In [3]:
# --- Colab setup: GPU runtime (Runtime > Change runtime type > T4 GPU) ---
REPO_URL = "https://github.com/gaurkhare/gaurav-eagv5-s13.git"
import os, sys, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/era_a13"          # data + results persist across notebooks
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    os.chdir("/content/repo")
else:
    WORK = os.path.abspath("..")                      # running locally from notebooks/
    os.chdir(WORK)
sys.path.insert(0, os.getcwd())
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATA_DIR, RESULTS = f"{WORK}/data", f"{WORK}/results"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no nvidia GPU"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
!pip -q install datasets transformers

In [5]:
from revllm.data import prepare
if not os.path.exists(f"{DATA_DIR}/train.bin"):
    vocab = prepare(DATA_DIR, train_tokens=51_000_000, val_tokens=1_000_000)
    print("vocab", vocab)
!ls -lh {DATA_DIR}

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3956 > 2048). Running this sequence through the model will result in indexing errors


wrote /content/drive/MyDrive/era_a13/data/val.bin: 1,000,000 tokens
  train: 5.0M / 51M tokens
  train: 10.9M / 51M tokens
  train: 15.4M / 51M tokens
  train: 20.1M / 51M tokens
  train: 25.7M / 51M tokens
  train: 30.9M / 51M tokens
  train: 35.7M / 51M tokens
  train: 40.1M / 51M tokens
  train: 45.8M / 51M tokens
  train: 50.6M / 51M tokens
wrote /content/drive/MyDrive/era_a13/data/train.bin: 51,000,000 tokens
vocab 32000
total 100M
-rw------- 1 root root  98M Sep 23 13:20 train.bin
-rw------- 1 root root 2.0M Sep 23 13:18 val.bin


In [6]:
import numpy as np
from transformers import AutoTokenizer
from revllm.data import TOKENIZER
tok = AutoTokenizer.from_pretrained(TOKENIZER)
arr = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r")
print(len(arr), "train tokens; max id", arr[:5_000_000].max())
print(tok.decode(arr[:200].tolist()))

51000000 train tokens; max id 31985
local producers are forced to remain non productive on that particular patented items. The standard of examination varies from country to country. In some countries like Netherlands, Germany, U.S.A. and Japan it is rigorously involving an extensive search for both novelty and obviousness among documents published in many countries, over a period of many years. But according to our existing patent law; examination is less rigorous involving for novelty only, and the extent of search is restricted. Term of patent protection shall, as laid down in section-14, be sixteen years from its date. But the term is not sufficient enough for the exploitation of the patent right. What constitutes infringement of patent isn’t defined in the Patents and Designs Act, 1911. Section-29(1) of the Act only says that, the patentee has a right to sue against the in
